In [33]:
import tensorflow as tf
import numpy as np
import pandas as pd

In [34]:
import numpy as np

class NeuralNetwork:
    '''
    layers should be a list where the number at index 0 should tell about the number of features your data has and further the number of neurons for each layer
    '''
    def __init__(self, layers):
        self.layers = len(layers) - 1
        # Tip: randn (normal distribution) is usually better than rand (uniform 0 to 1) 
        # for neural networks to avoid exploding/vanishing gradients early on.
        self.weights = [np.random.randn(layers[i], layers[i+1]) * 0.1 for i in range(len(layers) - 1)]
        self.activations = []
        self.z_values = []
        self.loss_history = [] # Renamed to avoid collision
    
    # Added 'self' parameter
    def relu(self, x):
        return np.maximum(0, x)
    
    # Added 'self' parameter
    def relu_derivative(self, x):
        return np.where(x > 0, 1, 0)
    
    # Renamed to avoid collision
    def calculate_loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    def forward_pass(self, x):
        self.activations = [x]
        self.z_values = []
        for i in range(self.layers):
            z = np.dot(self.activations[-1], self.weights[i])
            self.z_values.append(z)
            if i < self.layers - 1:
                a = self.relu(z)
            else:
                a = z 
            self.activations.append(a)
    
    def backward_pass(self, y_true, learning_rate):
        m = y_true.shape[0]
        delta = (2/m) * (self.activations[-1] - y_true)
        for i in reversed(range(self.layers)):
            current_activation = self.activations[i]
            grad_w = np.dot(current_activation.T, delta)
            if i > 0:
                prev_z = self.z_values[i-1]
                delta = (2/m) * np.dot(delta, self.weights[i].T) * self.relu_derivative(prev_z)
            self.weights[i] -= learning_rate * grad_w

    def train(self, X, y, epochs, learning_rate):
        self.loss_history = [] # Use renamed list
        for epoch in range(epochs):
            self.forward_pass(X)
            self.backward_pass(y, learning_rate)
            # Use renamed list and method
            current_loss = self.calculate_loss(y, self.activations[-1])
            self.loss_history.append(current_loss)
            
            # Optional: Print loss every 10 epochs to see it learning!
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Loss: {current_loss}")

In [35]:
class NeuralNetworkv1(tf.Module): # FIX 1: tf.Module, not tf.nn.Module
    def __init__(self, layers, learning_rate=0.01):
        super().__init__()
        self.layers = layers 
        
        # FIX 2: Pass shape as a list [in, out]
        self.weights = [
            tf.Variable(tf.random.normal(shape=[layers[i], layers[i+1]]) * 0.1, name=f'W_{i}') 
            for i in range(len(layers) - 1)
        ]
        
        # Initialize optimizer with the learning rate here
        self.optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
        self.loss_history = [] # To track loss over epochs

    def calculate_loss(self, y_true, y_pred):
        return tf.reduce_mean(tf.square(y_true - y_pred)) # Standard MSE
        #  return tf.reduce_sum(tf.square(y_true - y_pred)) / 2.0 

    # FIX 3: No more saving self.activations! Just return the final prediction.
    def forward_pass(self, x):
        a = x
        for i in range(len(self.layers) - 1):
            z = tf.matmul(a, self.weights[i])
            if i < len(self.layers) - 2:
                a = tf.nn.relu(z)
            else:
                a = z # Linear output
        return a
    
    def train(self, X, y, epochs):
        # Ensure inputs are float32 tensors (TF is strict about types)
        X = tf.cast(X, dtype=tf.float32)
        y = tf.cast(y, dtype=tf.float32)
        
        for epoch in range(epochs):
            # The 'tape' automatically watches any tf.Variable used inside this block
            with tf.GradientTape() as tape:
                predictions = self.forward_pass(X)
                loss = self.calculate_loss(y, predictions)
            # The tape calculates ALL the partial derivatives for you instantly
            gradients = tape.gradient(loss, self.weights)
            
            # The optimizer applies the W = W - (lr * gradient) update
            self.optimizer.apply_gradients(zip(gradients, self.weights))
            self.loss_history.append(loss.numpy())
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, Loss: {loss.numpy():.4f}")

In [36]:


# Create 100 samples with 4 features
X = np.random.normal(0, 1, (100, 4)) 

# Create a true relationship! 
# Let's make y equal to: (2 * feature_1) + (-1 * feature_2)
# We keep the shape as (100, 1)
y = (X[:, 0] * 2 + X[:, 1] * -1).reshape(100, 1)

# Define the neural network architecture
nn = NeuralNetwork(layers=[4, 3, 2, 1])
nnv1 = NeuralNetworkv1(layers=[4, 3, 2, 1], learning_rate=0.01)


# Train the model (you might need to drop the learning rate slightly 
# since we are passing all 100 samples in a single batch, which sums the gradients)
print("Training with custom NumPy implementation:")
nn.train(X, y, epochs=100, learning_rate=0.1)
print(f"Final Loss: {nn.loss_history[-1]}")

print("\nTraining with TensorFlow implementation:")
nnv1.train(X, y, epochs=100)
print("Final Loss:", nnv1.loss_history[-1])
# print(f"Final Loss: {nnv1.calculate_loss(y, nnv1.forward_pass(X)).numpy():.4f}")

Training with custom NumPy implementation:
Epoch 10, Loss: 5.525677413223243
Epoch 20, Loss: 5.525367226510724
Epoch 30, Loss: 5.525050466837001
Epoch 40, Loss: 5.524723585651582
Epoch 50, Loss: 5.5243850623148365
Epoch 60, Loss: 5.52403262962086
Epoch 70, Loss: 5.523663794943302
Epoch 80, Loss: 5.523275947621027
Epoch 90, Loss: 5.522866338749248
Epoch 100, Loss: 5.522432059976078
Final Loss: 5.522432059976078

Training with TensorFlow implementation:
Epoch 10, Loss: 5.5238
Epoch 20, Loss: 5.5235
Epoch 30, Loss: 5.5231
Epoch 40, Loss: 5.5228
Epoch 50, Loss: 5.5224
Epoch 60, Loss: 5.5219
Epoch 70, Loss: 5.5214
Epoch 80, Loss: 5.5209
Epoch 90, Loss: 5.5203
Epoch 100, Loss: 5.5196
Final Loss: 5.5195723


In [ ]:
# ==========================================
# 1. The Dataset (Trial by Combat Records)
# ==========================================
# Features: [Duncan's Reach Advantage, Armor Thickness, Opponent's Skill Level]
# Generating 1,000 mock battles
X = np.random.uniform(low=0.0, high=10.0, size=(1000, 3))

# Target: 1 if Ser Duncan wins, 0 if he loses. 
# Logic: He likely wins if his reach and armor outscale the opponent's skill.
y = ((X[:, 0] * 1.5 + X[:, 1]) > X[:, 2]).astype(int)

# ==========================================
# 2. Building the Architecture 
# ==========================================
# The Sequential API lets you stack layers exactly like your handwritten math
model = tf.keras.Sequential([
    # Layer 1: 8 neurons, ReLU activation. 
    # 'input_shape' replaces your manual weight initialization matrix dimensions.
    tf.keras.layers.Dense(8, activation='relu', input_shape=(3,), name="Hedge_Knight_Layer"),
    
    # Layer 2: 4 neurons, ReLU activation.
    tf.keras.layers.Dense(4, activation='relu', name="Squire_Layer"),
    
    # Layer 3 (Output): 1 neuron, Sigmoid activation.
    # Sigmoid squeezes the output between 0 and 1 (perfect for probability).
    tf.keras.layers.Dense(1, activation='sigmoid', name="Victory_Prediction") 
])

# View the matrix shapes and parameters you just created
model.summary()

# ==========================================
# 3. Compiling the Model
# ==========================================
# This replaces your manual gradient descent and loss calculation methods
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy', # The standard loss for binary (1 or 0) targets
    metrics=['accuracy']        # Let's track accuracy alongside the loss
)

# ==========================================
# 4. Training the Network (The Melee)
# ==========================================
print("\nCommencing Training...")

# The .fit() method handles the forward pass, backprop, and weight updates automatically
history = model.fit(
    X, y,
    epochs=50,
    batch_size=32,          # Keras automatically divides your gradients by the batch size!
    validation_split=0.2,   # Automatically holds back 20% of the data to test generalization
    verbose=1               # Prints a clean progress bar for every epoch
)

# ==========================================
# 5. Making Predictions
# ==========================================
# Let's test a new battle: [Reach=8.0, Armor=7.0, Opponent Skill=9.0]
new_battle = np.array([[8.0, 7.0, 9.0]])
prediction = model.predict(new_battle)

print(f"\nProbability of Ser Duncan winning: {prediction[0][0] * 100:.2f}%")